# Maryland Funding Pipeline — Step by Step Walkthrough
**Capstone INFO 588 | Built with INFO 570 techniques + Azure AI**

Run each cell one at a time. Read the output before moving to the next cell.

---
## Pipeline Overview
```
EXTRACT Stage 1  →  Call Knack API        →  Get 54 programs (name, type, URL)
EXTRACT Stage 2  →  Follow each URL       →  Fetch detail page HTML
BS4 SCAN         →  BeautifulSoup         →  Find all $ mentions (ground truth check)
TRANSFORM        →  Azure GPT-4o          →  Extract structured fields + classify all amounts
REVIEW CHECK     →  Python logic          →  Flag records with multiple/ambiguous amounts
LOAD             →  Save to JSON          →  Clean records + flagged records separated
```

## Target Output Schema (matches Runwei platform columns)
```
title                   → Program name
opportunity_type        → Grant / Loan / Tax Credit / Forgivable Loan / etc.
summary                 → 2 sentence max overview
description             → 5-10 sentences, full detail
sponsor                 → Entity offering the opportunity
sponsor_website         → Sponsor primary website URL
logo_url                → Logo image URL or 'No logo URL found'
direct_application_url  → Direct submission/application link
award_value             → Best single dollar amount or 'Varies' / 'Not specified'
cash_award              → Cash portion if separate from total award
award_amounts           → All amounts with context (audit trail)
date_posted             → MM-DD-YYYY or 'Not specified'
deadline                → MM-DD-YYYY or 'Rolling' — never empty
rolling                 → Yes / No
global_opportunity      → Yes / No
location                → Geographic focus
fee_required            → No | Yes - $X
cost_to_participate     → No | Yes - $X
equity_percentage       → No | Yes - X%
safe_note               → No | Yes - details
contact_names           → Contact names if available
contact_email           → Contact email if available
industry                → Industry field
tags                    → ['innovation', 'entrepreneurship', ...]
areas_of_focus          → ['Capital', 'Networks', 'Capacity Building']
eligibility             → Bullet point list
sdg_alignment           → ['SDG 8 Decent Work and Economic Growth', ...]
needs_review            → True if multiple/ambiguous amounts found
review_reason           → Why it was flagged
```

---
## Step 0 — Imports & Configuration

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
import time
import os

AZURE_ENDPOINT    = ''
AZURE_API_KEY     = ''
AZURE_DEPLOYMENT  = 'gpt4'
AZURE_API_VERSION = '2024-12-01-preview'

KNACK_URL = 'https://us-east-1-renderer-read.knack.com/v1/scenes/scene_1/views/view_17/records'
KNACK_HEADERS = {
    'X-Knack-Application-Id': '594ac6010f1b2d4e1e14a3b8',
    'X-Knack-REST-API-Key': 'renderer'
}

print('Libraries loaded and credentials set')

---
## EXTRACT Stage 1 — Call the Knack API


In [ ]:
params = {
    'format': 'both',
    'page': 1,
    'rows_per_page': 50,
    'sort_field': 'field_21',
    'sort_order': 'asc'
}

response = requests.get(KNACK_URL, headers=KNACK_HEADERS, params=params, timeout=10)
response.raise_for_status()
data = response.json()

print(f'Status code: {response.status_code}')
print(f'Total programs: {data["total_records"]}')
print(f'Total pages: {data["total_pages"]}')
print(f'Records on this page: {len(data["records"])}')

In [ ]:
programs = []

for record in data['records']:
    url_match = re.search(r'href="([^"]+)"', record.get('field_23', ''))
    detail_url = url_match.group(1) if url_match else None
    if not detail_url:
        continue
    programs.append({
        'id':       record.get('id', ''),
        'name':     record.get('field_21', '').strip(),
        'type':     record.get('field_22', '').strip(),
        'category': record.get('field_27', '').strip(),
        'url':      detail_url
    })

df_list = pd.DataFrame(programs)
print(f'Programs with valid URLs: {len(programs)}')
df_list[['name', 'type', 'category']].head(10)

---
## EXTRACT Stage 2 — Fetch a Detail Page
**INFO 570 connection:** `requests.get()` for web page fetching from Module 3.

In [ ]:
test_program = programs[0]

print(f'Program: {test_program["name"]}')
print(f'URL:     {test_program["url"]}')

page_response = requests.get(test_program['url'], timeout=10)
page_response.raise_for_status()
html = page_response.text

print(f'Page fetched — {len(html):,} characters of HTML')

---
## BS4 Dollar Scan — Ground Truth Check
BeautifulSoup scans the page and finds EVERY line containing `$`.
This is our ground truth — if Azure returns an amount not in this list, it hallucinated.

BeautifulSoup parsing + list comprehension 

In [ ]:
soup      = BeautifulSoup(html, 'html.parser')
full_text = soup.get_text(separator='\n')


dollar_lines = [
    line.strip()
    for line in full_text.split('\n')
    if '$' in line and line.strip()
]

print(f'Program: {test_program["name"]}')
print(f'Dollar mentions found: {len(dollar_lines)}')
print()

if dollar_lines:
    print('--- Lines containing $ ---')
    for i, line in enumerate(dollar_lines, 1):
        print(f'  [{i}] {line}')
    if len(dollar_lines) > 1:
        print()
        print('  Multiple dollar amounts found — Azure will classify each one.')
        print('   This record may be flagged for review.')
else:
    print(' No dollar amounts on this page — award_value will be "Not specified".')

---
## TRANSFORM — Azure GPT-4o Extracts + Classifies All Amounts
Azure extracts all fields AND classifies every dollar amount found.
For each amount it tells us: is this the individual award, total fund size, minimum, or something else?

**Why this matters:** A page might show `$5M total fund` and `up to $50,000 per award`.
Without classification, we'd pick the wrong number. Azure figures out which is which.

In [ ]:
dollar_context = '\n'.join([f'[{i+1}] {l}' for i, l in enumerate(dollar_lines)]) if dollar_lines else 'No dollar amounts found.'

system_prompt = """You are extracting structured data from a government funding opportunity page
to match the Runwei platform schema. Extract every field below.
If a field cannot be found on the page, use 'Not specified' — never invent data.

Fields to extract:

1.  title: Full program name.
2.  opportunity_type: Best match from: Grant, Loan, Tax Credit, Forgivable Loan, Accelerator,
    Incubator, Fellowship, Competition, Pro Bono, Internship, SaaS Credit, Stipend,
    Mentorship, Workshop, Legislative Initiative.
3.  summary: 2 sentences max. Key value and purpose of the opportunity.
4.  description: 5-10 sentences. Objectives, benefits, and eligibility overview.
5.  sponsor: The government agency or entity offering this opportunity.
6.  sponsor_website: The sponsor primary website URL. 'Not specified' if not found.
7.  logo_url: Logo image URL from the page. 'No logo URL found' if not present.
8.  direct_application_url: Direct application or submission link. 'Not specified' if not found.
9.  award_value: The single best dollar amount an individual applicant can receive.
    Use ONLY amounts from the verified dollar lines provided.
    If page shows total fund AND individual amount, use the individual amount.
    If only total fund size exists, write 'Varies'.
    If no dollar amount at all, write 'Not specified'. Never invent amounts.
10. cash_award: Cash portion if separate from total award. 'Not specified' if not found.
11. award_amounts: List of ALL dollar amounts from verified dollar lines with context:
    [{"amount": "$50,000", "context": "maximum individual grant award"}]
    Empty list if none found.
12. date_posted: Posting date MM-DD-YYYY. 'Not specified' if not found.
13. deadline: Application deadline MM-DD-YYYY. If rolling write 'Rolling'. Never empty.
14. rolling: 'Yes' if rolling basis. Otherwise 'No'.
15. global_opportunity: 'Yes' if globally available. Otherwise 'No'.
16. location: Geographic focus or eligible regions. 'Not specified' if not found.
17. fee_required: 'No' if no fee. If yes: 'Yes - $X'.
18. cost_to_participate: 'No' if no cost. If yes: 'Yes - $X'.
19. equity_percentage: 'No' if no equity. If yes: 'Yes - X%'.
20. safe_note: 'No' if no SAFE note. If yes: 'Yes - details'.
21. contact_names: Contact names if listed. 'Not specified' if none.
22. contact_email: Contact emails if listed. 'Not specified' if none.
23. industry: Industries relevant to this opportunity. 'Not specified' if not found.
24. tags: List of keyword tags.
25. areas_of_focus: List from Capital, Networks, Capacity Building. Empty list if none apply.
26. eligibility: List of eligibility requirements as strings.
27. sdg_alignment: Applicable UN SDGs (e.g. 'SDG 8 Decent Work and Economic Growth').

Return valid JSON only."""

user_prompt = f"""Program Name: {test_program['name']}

Verified dollar lines from page (BeautifulSoup scan):
{dollar_context}

HTML Content (first 8000 characters):
{html[:8000]}

Extract as JSON. For award_amounts, classify each verified dollar line.
For award_value, pick the individual award amount — not total fund size."""

api_url   = f"{AZURE_ENDPOINT}/openai/deployments/{AZURE_DEPLOYMENT}/chat/completions?api-version={AZURE_API_VERSION}"
az_headers = {'Content-Type': 'application/json', 'api-key': AZURE_API_KEY}

payload = {
    'messages': [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_prompt}
    ],
    'temperature': 0.1,
    'max_tokens': 2000,
    'response_format': {'type': 'json_object'}
}

az_response = requests.post(api_url, headers=az_headers, json=payload, timeout=30)
az_response.raise_for_status()

result    = az_response.json()
extracted = json.loads(result['choices'][0]['message']['content'])
tokens    = result['usage']['total_tokens']

print(f'Extraction complete — {tokens} tokens used')
print()
print(f'award_value:   {extracted.get("award_value", "MISSING")}')
print(f'award_amounts: {json.dumps(extracted.get("award_amounts", []), indent=2)}')
print(f'deadline:      {extracted.get("deadline", "MISSING")}')
print()
print('Full extraction:')
print(json.dumps(extracted, indent=2))

---
## REVIEW CHECK — Flag Ambiguous Records
After extraction, we check if this record needs human review.
The system handles what it can — escalates what it can't.

**Flagged when:**
- Multiple dollar amounts found AND Azure couldn't confidently pick one
- Award value is 'Varies' but multiple specific amounts exist on the page
- More than 2 distinct dollar amounts found

In [ ]:
award_amounts = extracted.get('award_amounts', [])
award_value   = extracted.get('award_value', 'Not specified')

needs_review  = False
review_reason = ''

# Rule 1: More than 2 distinct dollar amounts found
if len(award_amounts) > 2:
    needs_review  = True
    review_reason = f'{len(award_amounts)} dollar amounts found human should confirm correct award_value'

# Rule 2: Award value is Varies but specific amounts exist
elif award_value == 'Varies' and len(dollar_lines) > 0:
    needs_review  = True
    review_reason = 'Award marked Varies but dollar amounts exist on page — verify manually'

# Rule 3: Multiple amounts AND award_value looks like a fund total (contains M or B)
elif len(award_amounts) > 1 and any(x in award_value for x in ['M', 'B', 'million', 'billion']):
    needs_review  = True
    review_reason = 'Award value may be fund total, not individual award — verify manually'

if needs_review:
    print(f'FLAGGED FOR REVIEW')
    print(f'Reason: {review_reason}')
    print(f'award_value set to: {award_value}')
    print(f'All amounts found:')
    for a in award_amounts:
        print(f'{a["amount"]} ({a["context"]})')
else:
    print(f'No review needed')
    print(f'award_value: {award_value}')
    if award_amounts:
        print(f'Amounts classified: {len(award_amounts)}')

---
## LOAD — Save the Result
Clean records go to `maryland_results.json`.
Flagged records go to `maryland_review_queue.json` for human review.

In [ ]:
final_record = {
    'id':                       test_program['id'],
    'source_url':               test_program['url'],
    'state':                    'Maryland',
    # Core identity
    'title':                    extracted.get('title', test_program['name']),
    'opportunity_type':         extracted.get('opportunity_type', test_program['type']),
    'summary':                  extracted.get('summary', 'Not specified'),
    'description':              extracted.get('description', 'Not specified'),
    # Sponsor info
    'sponsor':                  extracted.get('sponsor', 'Not specified'),
    'sponsor_website':          extracted.get('sponsor_website', 'Not specified'),
    'logo_url':                 extracted.get('logo_url', 'No logo URL found'),
    'direct_application_url':   extracted.get('direct_application_url', 'Not specified'),
    # Award info
    'award_value':              award_value,
    'cash_award':               extracted.get('cash_award', 'Not specified'),
    'award_amounts':            award_amounts,
    # Dates
    'date_posted':              extracted.get('date_posted', 'Not specified'),
    'deadline':                 extracted.get('deadline', 'Not specified'),
    'rolling':                  extracted.get('rolling', 'No'),
    # Geography
    'global_opportunity':       extracted.get('global_opportunity', 'No'),
    'location':                 extracted.get('location', 'Not specified'),
    # Compliance flags
    'fee_required':             extracted.get('fee_required', 'No'),
    'cost_to_participate':      extracted.get('cost_to_participate', 'No'),
    'equity_percentage':        extracted.get('equity_percentage', 'No'),
    'safe_note':                extracted.get('safe_note', 'No'),
    # Contact
    'contact_names':            extracted.get('contact_names', 'Not specified'),
    'contact_email':            extracted.get('contact_email', 'Not specified'),
    # Classification
    'industry':                 extracted.get('industry', 'Not specified'),
    'tags':                     extracted.get('tags', []),
    'areas_of_focus':           extracted.get('areas_of_focus', []),
    'eligibility':              extracted.get('eligibility', []),
    'sdg_alignment':            extracted.get('sdg_alignment', []),
    # Pipeline metadata
    'needs_review':             needs_review,
    'review_reason':            review_reason,
    'dollar_scan':              dollar_lines,
    'tokens_used':              tokens
}

# Save to main results
with open('maryland_test_result.json', 'w') as f:
    json.dump([final_record], f, indent=2)

# If flagged — also save to review queue
if needs_review:
    with open('maryland_review_queue.json', 'a') as f:
        f.write(json.dumps(final_record, indent=2) + ',\n')
    print('Saved to maryland_review_queue.json (needs human review)')
else:
    print('Saved to maryland_test_result.json (clean)')

print()
print(json.dumps(final_record, indent=2))

---
## Pipeline Summary
```
EXTRACT Stage 1  →  Knack API            →  54 programs retrieved
EXTRACT Stage 2  →  requests.get()       →  Detail page HTML fetched
BS4 SCAN         →  BeautifulSoup        →  All $ amounts identified (ground truth)
TRANSFORM        →  Azure GPT-4o         →  All fields extracted + amounts classified
REVIEW CHECK     →  Python logic         →  Ambiguous records flagged automatically
LOAD             →  JSON files           →  Clean → results | Flagged → review_queue
```

## Next Steps
1. Confirm output matches sponsor platform format
2. Loop through all 54 programs (change `programs[0]` to a loop)
3. Review flagged records in `maryland_review_queue.json`
4. Design PostgreSQL schema and load the data